<a href="https://colab.research.google.com/github/AnugrahaSaji/AnugrahaSaji/blob/main/Aadhar_authentication_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install qiskit qiskit-aer pycryptodome openpyxl pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.5 MB/s eta 0:00:00


In [3]:
from google.colab import files
uploaded = files.upload()

Saving 50_voters_with_aadhaar_dataset (4).xlsx to 50_voters_with_aadhaar_dataset (4).xlsx


In [4]:
import pandas as pd

file_path = "50_voters_with_aadhaar_dataset (4).xlsx"
df = pd.read_excel(file_path)

print(df.head())

   Voter ID       Name Department  Aadhaar Number  Has Voted
0  VOTER001  Student_1        CSE    100000000001      False
1  VOTER002  Student_2         IT    100000000002      False
2  VOTER003  Student_3         IT    100000000003      False
3  VOTER004  Student_4        CSE    100000000004      False
4  VOTER005  Student_5        CSE    100000000005      False


In [ ]:
# ==========================================
# QUANTUM E-VOTING WITH REAL CLIENT-SERVER LOGIC
# ==========================================

from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
import hashlib
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes

print("\n====== CLIENT-SERVER QUANTUM E-VOTING ======\n")

# -----------------------------
# CREATE SIMPLE DATASET
# -----------------------------
database = {}
for i in range(1, 6):  # 5 voters for demo
    voter_id = f"VOTER{i:03d}"
    database[voter_id] = {
        "name": f"Student_{i}",
        "aadhaar": str(100000000000 + i),
        "has_voted": False
    }

votes_storage = []

# ====================================================
# SERVER SIDE FUNCTIONS
# ====================================================

def server_generate_quantum_key():
    qc = QuantumCircuit(2,2)
    qc.h(0)
    qc.cx(0,1)
    qc.measure([0,1],[0,1])

    sim = Aer.get_backend('aer_simulator')
    qc = transpile(qc, sim)
    result = sim.run(qc, shots=1).result()
    counts = result.get_counts()
    return list(counts.keys())[0]

def server_create_challenge(hashed_key):
    challenge = get_random_bytes(16)
    cipher = AES.new(hashed_key, AES.MODE_GCM)
    ciphertext, tag = cipher.encrypt_and_digest(challenge)

    return {
        "ciphertext": ciphertext,
        "nonce": cipher.nonce,
        "tag": tag,
        "original_challenge": challenge  # server keeps original
    }

def server_verify_response(original_challenge, client_response):
    return original_challenge == client_response


# ====================================================
# CLIENT SIDE FUNCTION
# ====================================================

def client_decrypt_challenge(hashed_key, ciphertext, nonce, tag):
    cipher = AES.new(hashed_key, AES.MODE_GCM, nonce=nonce)
    decrypted = cipher.decrypt_and_verify(ciphertext, tag)
    return decrypted


# ====================================================
# MAIN AUTHENTICATION FLOW
# ====================================================

while True:
    print("\n--- LOGIN ---")
    voter_id = input("Enter Voter ID (or EXIT): ")

    if voter_id == "EXIT":
        break

    aadhaar_input = input("Enter Aadhaar: ")

    if voter_id not in database:
        print("❌ Invalid Voter")
        continue

    if database[voter_id]["aadhaar"] != aadhaar_input:
        print("❌ Aadhaar Verification Failed")
        continue

    if database[voter_id]["has_voted"]:
        print("❌ Already Voted")
        continue

    print("✅ Identity Verified")

    # -------- QUANTUM KEY (SHARED) --------
    quantum_key = server_generate_quantum_key()
    hashed_key = hashlib.sha256(quantum_key.encode()).digest()

    print("🔐 Quantum Key Established")

    # -------- SERVER CREATES CHALLENGE --------
    challenge_packet = server_create_challenge(hashed_key)

    print("📤 Server sent encrypted challenge")

    # -------- CLIENT DECRYPTS --------
    client_response = client_decrypt_challenge(
        hashed_key,
        challenge_packet["ciphertext"],
        challenge_packet["nonce"],
        challenge_packet["tag"]
    )

    print("📥 Client decrypted challenge and sent response")

    # -------- SERVER VERIFIES --------
    if server_verify_response(
        challenge_packet["original_challenge"],
        client_response
    ):
        print("🔓 Authentication Successful")
    else:
        print("❌ Authentication Failed")
        continue

    # -------- CAST VOTE --------
    print("\nCandidates: A, B, C")
    vote = input("Enter your vote: ").upper()

    if vote not in ["A", "B", "C"]:
        print("❌ Invalid Candidate")
        continue

    cipher_vote = AES.new(hashed_key, AES.MODE_GCM)
    encrypted_vote, tag_vote = cipher_vote.encrypt_and_digest(vote.encode())

    votes_storage.append(encrypted_vote.hex())
    database[voter_id]["has_voted"] = True

    print("🗳 Vote Stored (Encrypted):", encrypted_vote.hex())

    # Destroy session
    quantum_key = None
    hashed_key = None

print("\n====== SESSION ENDED ======")
print("Encrypted Votes:", votes_storage)


====== CLIENT-SERVER QUANTUM E-VOTING ======


--- LOGIN ---
Enter Voter ID (or EXIT): VOTER001
Enter Aadhaar: 100000000001
✅ Identity Verified
🔐 Quantum Key Established
📤 Server sent encrypted challenge
📥 Client decrypted challenge and sent response
🔓 Authentication Successful

Candidates: A, B, C
Enter your vote: A
🗳 Vote Stored (Encrypted): 55

--- LOGIN ---
Enter Voter ID (or EXIT): VOTER001
Enter Aadhaar: 100000000001
❌ Already Voted

--- LOGIN ---
